# PyTorch Tutorial 42: Coding Agents from Scratch (The FAANG Standard)

Tutorial 31 covered generic agentic systems — ReAct, multi-agent orchestration, memory. This tutorial focuses on **code-specific agents**: the systems that power GitHub Copilot, Cursor, Devin, and xAI's coding assistants.

We build everything from scratch: AST parsing, sandboxed execution, self-debugging loops, and a complete coding agent that can plan, write, test, and fix code.

## Learning Objectives
1. **Parse and understand code** with Python's AST module — extract functions, find bugs, compute complexity
2. **Build a coding tool framework** — file I/O, terminal execution, code search, test running
3. **Implement sandboxed code execution** — safe subprocess isolation with timeouts and resource limits
4. **Build a self-debugging agent** — the generate-execute-fix loop that makes coding agents powerful
5. **Evaluate coding agents** — pass@k metrics, functional correctness, solve rates

**Prerequisites**: Tutorial 31 (ReAct agents, tool use), Tutorial 43 (pass@k, sandboxing)

---

## 1. Vocabulary First

- **AST (Abstract Syntax Tree)**: Tree representation of code structure. Python's `ast` module parses source code into a tree of nodes (FunctionDef, ClassDef, Assign, etc.).
- **Code Generation**: Producing code from natural language descriptions. Types: function completion, docstring-to-code, bug fixing.
- **Self-Debugging**: The agent generates code, executes it, observes errors, and iteratively fixes them. Key innovation over one-shot generation.
- **Sandboxed Execution**: Running untrusted code in an isolated environment with resource limits (CPU, memory, network).
- **Plan-and-Solve**: Decompose a complex task into subtasks, solve each, integrate. Better than ReAct for multi-step coding.
- **Cyclomatic Complexity**: Number of independent paths through code. Measures code complexity (McCabe metric).
- **pass@k**: Probability that at least 1 of k generated solutions passes all tests (see Tutorial 43).

### Generic Agents vs Coding Agents

| Aspect | Generic Agent (Tutorial 31) | Coding Agent (This Tutorial) |
|--------|---------------------------|-----------------------------|
| Tools | Calculator, Search, Wikipedia | File I/O, Terminal, Test Runner, AST |
| Verification | LLM-as-judge | Execute code and check output |
| Safety | Rate limiting, content filter | Sandboxed execution, resource limits |
| Feedback | Text observation | Compiler errors, test results, stack traces |
| Iteration | Usually 1-3 turns | Often 3-10 attempts to pass tests |

In [ ]:
import ast
import subprocess
import tempfile
import textwrap
import time
import re
import os
import json
import hashlib
from abc import ABC, abstractmethod
from dataclasses import dataclass, field
from typing import List, Dict, Tuple, Optional, Any
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

np.random.seed(42)
print("Ready for Coding Agents!")

---

## 2. Part 1: AST Parsing and Code Understanding

Before an agent can write code, it needs to **understand** code. Python's `ast` module gives us structural understanding.

### FAANG Interview Question

**Q: "Why use AST instead of regex for code analysis?"**

**A**: Regex operates on text — it can't distinguish between a function definition and a string containing 'def'. AST operates on the **parsed structure** of code, so it understands scope, nesting, and semantics. For example, finding all undefined variables requires understanding scope chains, which regex can't do. AST also handles multi-line statements, decorators, and nested functions correctly.

In [ ]:
class CodeAnalyzer:
    """Static analysis tool using Python's AST module.
    
    Extracts structural information from Python source code:
    functions, classes, complexity, call graphs, and potential bugs.
    """
    
    def __init__(self, source_code: str):
        self.source = source_code
        self.tree = ast.parse(source_code)
    
    def extract_functions(self) -> List[Dict[str, Any]]:
        """Extract all function definitions with metadata."""
        functions = []
        for node in ast.walk(self.tree):
            if isinstance(node, ast.FunctionDef):
                args = [a.arg for a in node.args.args]
                docstring = ast.get_docstring(node) or ""
                functions.append({
                    "name": node.name,
                    "args": args,
                    "line": node.lineno,
                    "docstring": docstring[:100],
                    "n_lines": node.end_lineno - node.lineno + 1 if node.end_lineno else 0
                })
        return functions
    
    def extract_classes(self) -> List[Dict[str, Any]]:
        """Extract all class definitions with their methods."""
        classes = []
        for node in ast.walk(self.tree):
            if isinstance(node, ast.ClassDef):
                methods = [n.name for n in node.body if isinstance(n, ast.FunctionDef)]
                classes.append({
                    "name": node.name,
                    "methods": methods,
                    "line": node.lineno,
                    "n_methods": len(methods)
                })
        return classes
    
    def count_complexity(self) -> int:
        """Compute cyclomatic complexity (simplified McCabe metric).
        
        Complexity = 1 + number of decision points
        (if, elif, for, while, except, and, or, assert)
        """
        complexity = 1  # base complexity
        decision_nodes = (
            ast.If, ast.For, ast.While, ast.ExceptHandler,
            ast.Assert, ast.BoolOp
        )
        for node in ast.walk(self.tree):
            if isinstance(node, decision_nodes):
                complexity += 1
        return complexity
    
    def find_undefined_vars(self) -> List[str]:
        """Find variables used before definition (simple scope analysis)."""
        defined = set()
        used_before_def = []
        
        for node in ast.walk(self.tree):
            if isinstance(node, ast.Assign):
                for target in node.targets:
                    if isinstance(target, ast.Name):
                        defined.add(target.id)
            elif isinstance(node, ast.FunctionDef):
                defined.add(node.name)
                for arg in node.args.args:
                    defined.add(arg.arg)
            elif isinstance(node, ast.Name) and isinstance(node.ctx, ast.Load):
                # Built-in names to ignore
                builtins = {'print', 'len', 'range', 'int', 'str', 'float', 
                           'list', 'dict', 'set', 'True', 'False', 'None',
                           'isinstance', 'type', 'sorted', 'enumerate',
                           'zip', 'map', 'filter', 'sum', 'min', 'max', 'abs'}
                if node.id not in defined and node.id not in builtins:
                    used_before_def.append(node.id)
        
        return list(set(used_before_def))
    
    def get_call_graph(self) -> Dict[str, List[str]]:
        """Build a call graph: which functions call which."""
        call_graph = {}
        
        for node in ast.walk(self.tree):
            if isinstance(node, ast.FunctionDef):
                calls = []
                for child in ast.walk(node):
                    if isinstance(child, ast.Call):
                        if isinstance(child.func, ast.Name):
                            calls.append(child.func.id)
                        elif isinstance(child.func, ast.Attribute):
                            calls.append(child.func.attr)
                call_graph[node.name] = list(set(calls))
        
        return call_graph


# Test on sample code
sample_code = '''
class Calculator:
    """A simple calculator."""
    
    def __init__(self):
        self.history = []
    
    def add(self, a, b):
        """Add two numbers."""
        result = a + b
        self.history.append(result)
        return result
    
    def divide(self, a, b):
        """Divide a by b with error handling."""
        if b == 0:
            raise ValueError("Cannot divide by zero")
        return a / b

def fibonacci(n):
    """Compute fibonacci number."""
    if n <= 1:
        return n
    return fibonacci(n - 1) + fibonacci(n - 2)
'''

analyzer = CodeAnalyzer(sample_code)
print("Code Analysis Results:")
print("=" * 50)
print(f"\nFunctions: {analyzer.extract_functions()}")
print(f"\nClasses: {analyzer.extract_classes()}")
print(f"\nCyclomatic Complexity: {analyzer.count_complexity()}")
print(f"\nCall Graph: {analyzer.get_call_graph()}")

---

## 3. Part 2: Coding Tool Framework

A coding agent needs tools to interact with the environment. These are different from generic agent tools.

### FAANG Interview Question

**Q: "What safety measures do you need for a code-executing agent?"**

**A**: Five layers of safety:
1. **Input validation**: Check code for dangerous patterns before execution
2. **Process isolation**: Run in subprocess (not in the agent's process)
3. **Resource limits**: CPU timeout, memory cap, disk write limits
4. **Network isolation**: Block outbound network access
5. **Filesystem sandboxing**: Restrict to a temp directory, no access to host filesystem

In [ ]:
@dataclass
class ToolResult:
    """Standardized output from any tool execution."""
    success: bool
    output: str
    error: str = ""
    execution_time_ms: float = 0.0


class BaseTool(ABC):
    """Abstract base class for coding agent tools."""
    
    @property
    @abstractmethod
    def name(self) -> str:
        pass
    
    @property
    @abstractmethod
    def description(self) -> str:
        pass
    
    @abstractmethod
    def execute(self, **kwargs) -> ToolResult:
        pass


class FileReadTool(BaseTool):
    """Read a file from the workspace."""
    name = "file_read"
    description = "Read the contents of a file"
    
    def __init__(self, workspace_dir: str):
        self.workspace = Path(workspace_dir)
    
    def execute(self, path: str) -> ToolResult:
        target = self.workspace / path
        if not target.exists():
            return ToolResult(False, "", f"File not found: {path}")
        # Security: ensure path is within workspace
        if not str(target.resolve()).startswith(str(self.workspace.resolve())):
            return ToolResult(False, "", "Access denied: path outside workspace")
        content = target.read_text()
        return ToolResult(True, content)


class FileWriteTool(BaseTool):
    """Write content to a file in the workspace."""
    name = "file_write"
    description = "Write content to a file"
    
    def __init__(self, workspace_dir: str):
        self.workspace = Path(workspace_dir)
    
    def execute(self, path: str, content: str) -> ToolResult:
        target = self.workspace / path
        if not str(target.resolve()).startswith(str(self.workspace.resolve())):
            return ToolResult(False, "", "Access denied: path outside workspace")
        target.parent.mkdir(parents=True, exist_ok=True)
        target.write_text(content)
        return ToolResult(True, f"Written {len(content)} chars to {path}")


class CodeSearchTool(BaseTool):
    """Search for patterns in workspace files (grep-like)."""
    name = "code_search"
    description = "Search for a regex pattern in workspace files"
    
    def __init__(self, workspace_dir: str):
        self.workspace = Path(workspace_dir)
    
    def execute(self, pattern: str, file_glob: str = "*.py") -> ToolResult:
        results = []
        for filepath in self.workspace.glob(f"**/{file_glob}"):
            try:
                content = filepath.read_text()
                for i, line in enumerate(content.splitlines(), 1):
                    if re.search(pattern, line):
                        rel_path = filepath.relative_to(self.workspace)
                        results.append(f"{rel_path}:{i}: {line.strip()}")
            except (UnicodeDecodeError, PermissionError):
                continue
        
        if results:
            return ToolResult(True, "\n".join(results[:20]))
        return ToolResult(True, "No matches found")


class ToolRegistry:
    """Registry of available tools with schema generation."""
    
    def __init__(self):
        self._tools: Dict[str, BaseTool] = {}
    
    def register(self, tool: BaseTool):
        self._tools[tool.name] = tool
    
    def get(self, name: str) -> Optional[BaseTool]:
        return self._tools.get(name)
    
    def list_tools(self) -> List[str]:
        return [f"{t.name}: {t.description}" for t in self._tools.values()]


# Demo
with tempfile.TemporaryDirectory() as workspace:
    registry = ToolRegistry()
    registry.register(FileReadTool(workspace))
    registry.register(FileWriteTool(workspace))
    registry.register(CodeSearchTool(workspace))
    
    # Write a file
    write_tool = registry.get("file_write")
    result = write_tool.execute(path="hello.py", content="def greet(name):\n    return f'Hello, {name}!'\n")
    print(f"Write: {result.output}")
    
    # Read it back
    read_tool = registry.get("file_read")
    result = read_tool.execute(path="hello.py")
    print(f"Read: {result.output}")
    
    # Search
    search_tool = registry.get("code_search")
    result = search_tool.execute(pattern="def \\w+")
    print(f"Search: {result.output}")
    
    print(f"\nAvailable tools: {registry.list_tools()}")

---

## 4. Part 3: Sandboxed Code Execution

The most critical component. Running untrusted code safely requires multiple layers of protection.

In [ ]:
@dataclass
class ExecutionResult:
    """Result of sandboxed code execution."""
    stdout: str
    stderr: str
    return_code: int
    runtime_ms: float
    timed_out: bool = False
    passed: bool = False


class CodeSandbox:
    """Safe Python code execution with resource limits.
    
    Security measures:
    - Subprocess isolation (no shared memory with agent)
    - Timeout enforcement (default 10s)
    - Temp directory isolation
    - Blocked dangerous patterns checked before execution
    """
    
    BLOCKED_PATTERNS = [
        r'import\s+socket',
        r'import\s+requests',
        r'import\s+urllib',
        r'__import__',
        r'exec\s*\(',
        r'compile\s*\(',
        r'open\s*\(.*/etc/',
        r'shutil\.rmtree',
    ]
    
    def __init__(self, timeout: int = 10):
        self.timeout = timeout
    
    def _check_safety(self, code: str) -> Tuple[bool, str]:
        """Pre-execution safety check for dangerous patterns."""
        for pattern in self.BLOCKED_PATTERNS:
            if re.search(pattern, code):
                return False, f"Blocked pattern detected: {pattern}"
        return True, ""
    
    def execute(self, code: str) -> ExecutionResult:
        """Execute Python code in an isolated subprocess."""
        safe, reason = self._check_safety(code)
        if not safe:
            return ExecutionResult("", reason, 1, 0.0)
        
        with tempfile.NamedTemporaryFile(
            mode='w', suffix='.py', delete=False
        ) as f:
            f.write(code)
            f.flush()
            script_path = f.name
        
        try:
            start = time.time()
            result = subprocess.run(
                ['python3', script_path],
                capture_output=True, text=True,
                timeout=self.timeout,
                cwd=tempfile.gettempdir()
            )
            runtime = (time.time() - start) * 1000
            
            return ExecutionResult(
                stdout=result.stdout[:5000],
                stderr=result.stderr[:5000],
                return_code=result.returncode,
                runtime_ms=runtime,
                passed=(result.returncode == 0)
            )
        except subprocess.TimeoutExpired:
            return ExecutionResult(
                "", f"Timeout after {self.timeout}s", 1, 
                self.timeout * 1000, timed_out=True
            )
        finally:
            os.unlink(script_path)
    
    def execute_with_tests(self, code: str, test_code: str) -> ExecutionResult:
        """Execute code with test cases appended."""
        full_code = code + "\n\n" + test_code
        return self.execute(full_code)


# Test the sandbox
sandbox = CodeSandbox(timeout=5)

# Safe code
print("Test 1: Safe code")
result = sandbox.execute("print('Hello from sandbox!')\nprint(2 + 2)")
print(f"  Output: {result.stdout.strip()}")
print(f"  Passed: {result.passed}, Time: {result.runtime_ms:.0f}ms")

# Code with error
print("\nTest 2: Code with error")
result = sandbox.execute("x = 1 / 0")
print(f"  Error: {result.stderr.strip()[-80:]}")
print(f"  Passed: {result.passed}")

# Blocked dangerous code
print("\nTest 3: Dangerous code (blocked)")
result = sandbox.execute("import socket; s = socket.socket()")
print(f"  Error: {result.stderr}")
print(f"  Passed: {result.passed}")

# Code with tests
print("\nTest 4: Code with test cases")
code = "def add(a, b):\n    return a + b\n"
tests = "assert add(1, 2) == 3\nassert add(-1, 1) == 0\nprint('All tests passed!')\n"
result = sandbox.execute_with_tests(code, tests)
print(f"  Output: {result.stdout.strip()}")
print(f"  Passed: {result.passed}")

---

## 5. Part 4: Self-Debugging Loop

This is the **core innovation** of coding agents. Instead of one-shot generation, the agent:
1. **Generates** initial code
2. **Executes** it in the sandbox
3. **Observes** the error (if any)
4. **Fixes** the code based on the error
5. **Repeats** until tests pass or max attempts reached

### FAANG Interview Question

**Q: "Design a self-debugging agent for code generation. What's the architecture?"**

**A**: Three components: (1) **Generator** — produces code from task description + context. (2) **Executor** — runs code in sandbox, captures output/errors. (3) **Debugger** — takes (code, error, task) and produces fixed code. The loop: generate -> execute -> if error: debug -> execute -> repeat. Key design decisions: max attempts (prevent infinite loops), error parsing (extract relevant stack trace lines), context management (what to include in the fix prompt — full code vs diff vs just the error).

In [ ]:
@dataclass
class AgentStep:
    """Single step in agent execution."""
    thought: str
    action: str
    code: str
    result: Optional[ExecutionResult] = None
    timestamp: float = 0.0


@dataclass
class AgentResult:
    """Final result of agent execution."""
    code: str
    passed: bool
    steps: List[AgentStep] = field(default_factory=list)
    attempts: int = 0
    total_time_ms: float = 0.0


class SelfDebugAgent:
    """Coding agent with self-debugging capability.
    
    Uses template-based generation for demo purposes.
    In production, replace generate_code() and fix_code() with LLM calls.
    """
    
    def __init__(self, sandbox: CodeSandbox, max_attempts: int = 5):
        self.sandbox = sandbox
        self.max_attempts = max_attempts
        # Template solutions for demo (in production: LLM generates these)
        self._solutions = {
            "fibonacci": [
                # Attempt 1: has a bug (wrong base case)
                "def fibonacci(n):\n    if n <= 0:\n        return 0\n    return fibonacci(n-1) + fibonacci(n-2)\n",
                # Attempt 2: fix base case
                "def fibonacci(n):\n    if n <= 0:\n        return 0\n    if n == 1:\n        return 1\n    return fibonacci(n-1) + fibonacci(n-2)\n",
            ],
            "is_palindrome": [
                # Attempt 1: doesn't handle case sensitivity
                "def is_palindrome(s):\n    return s == s[::-1]\n",
                # Attempt 2: handle case + spaces
                "def is_palindrome(s):\n    cleaned = s.lower().replace(' ', '')\n    return cleaned == cleaned[::-1]\n",
            ],
            "two_sum": [
                # Attempt 1: brute force with off-by-one
                "def two_sum(nums, target):\n    for i in range(len(nums)):\n        for j in range(len(nums)):\n            if nums[i] + nums[j] == target:\n                return [i, j]\n    return []\n",
                # Attempt 2: fix (j should start at i+1)
                "def two_sum(nums, target):\n    for i in range(len(nums)):\n        for j in range(i + 1, len(nums)):\n            if nums[i] + nums[j] == target:\n                return [i, j]\n    return []\n",
            ],
        }
    
    def _identify_task(self, description: str) -> str:
        """Map task description to a template key."""
        desc_lower = description.lower()
        for key in self._solutions:
            if key.replace('_', ' ') in desc_lower or key in desc_lower:
                return key
        return "generic"
    
    def _generate_code(self, task: str, attempt: int, error: str = "") -> str:
        """Generate or fix code (template-based for demo).
        
        In production: LLM call with (task, previous_code, error) as context.
        """
        task_key = self._identify_task(task)
        solutions = self._solutions.get(task_key, [])
        
        if attempt < len(solutions):
            return solutions[attempt]
        elif solutions:
            return solutions[-1]  # Return best known solution
        return f"# Could not generate code for: {task}\npass\n"
    
    def solve(self, task: str, test_code: str) -> AgentResult:
        """Solve a coding task with self-debugging.
        
        Args:
            task: natural language task description
            test_code: test cases to validate the solution
        Returns:
            AgentResult with final code, pass/fail, and step trace
        """
        start_time = time.time()
        steps = []
        last_error = ""
        
        for attempt in range(self.max_attempts):
            # Generate or fix code
            if attempt == 0:
                thought = f"Generating initial solution for: {task}"
            else:
                thought = f"Fixing code based on error: {last_error[:100]}"
            
            code = self._generate_code(task, attempt, last_error)
            
            # Execute with tests
            exec_result = self.sandbox.execute_with_tests(code, test_code)
            
            step = AgentStep(
                thought=thought,
                action="generate" if attempt == 0 else "fix",
                code=code,
                result=exec_result,
                timestamp=time.time() - start_time
            )
            steps.append(step)
            
            if exec_result.passed:
                total_time = (time.time() - start_time) * 1000
                return AgentResult(
                    code=code, passed=True, steps=steps,
                    attempts=attempt + 1, total_time_ms=total_time
                )
            
            last_error = exec_result.stderr
        
        total_time = (time.time() - start_time) * 1000
        return AgentResult(
            code=code, passed=False, steps=steps,
            attempts=self.max_attempts, total_time_ms=total_time
        )


# Demo the self-debugging agent
agent = SelfDebugAgent(sandbox=CodeSandbox(timeout=5), max_attempts=5)

tasks = [
    ("fibonacci", "assert fibonacci(0) == 0\nassert fibonacci(1) == 1\nassert fibonacci(5) == 5\nassert fibonacci(10) == 55\nprint('All fibonacci tests passed!')\n"),
    ("is_palindrome", "assert is_palindrome('racecar') == True\nassert is_palindrome('Race Car') == True\nassert is_palindrome('hello') == False\nprint('All palindrome tests passed!')\n"),
    ("two_sum", "assert two_sum([2, 7, 11, 15], 9) == [0, 1]\nassert two_sum([3, 2, 4], 6) == [1, 2]\nprint('All two_sum tests passed!')\n"),
]

print("Self-Debugging Agent Demo")
print("=" * 60)

all_results = []
for task_name, tests in tasks:
    result = agent.solve(task_name, tests)
    all_results.append(result)
    
    status = "PASS" if result.passed else "FAIL"
    print(f"\nTask: {task_name}")
    print(f"  Status: {status}")
    print(f"  Attempts: {result.attempts}")
    print(f"  Time: {result.total_time_ms:.0f}ms")
    
    for i, step in enumerate(result.steps):
        passed = step.result.passed if step.result else False
        print(f"  Step {i+1}: [{step.action}] {'PASS' if passed else 'FAIL'}")
        if not passed and step.result:
            error_lines = step.result.stderr.strip().split('\n')
            if error_lines:
                print(f"    Error: {error_lines[-1][:80]}")

In [ ]:
# Visualize self-debugging attempts
fig, axes = plt.subplots(1, 3, figsize=(16, 5))

# Attempts per task
task_names = ['fibonacci', 'is_palindrome', 'two_sum']
attempts = [r.attempts for r in all_results]
passed = [r.passed for r in all_results]
colors = ['green' if p else 'red' for p in passed]

axes[0].bar(task_names, attempts, color=colors, alpha=0.7, edgecolor='black')
axes[0].set_title('Attempts Until Pass', fontsize=12, fontweight='bold')
axes[0].set_ylabel('Number of Attempts')
axes[0].axhline(y=1, color='gray', linestyle='--', alpha=0.3, label='One-shot')
axes[0].legend()
axes[0].grid(True, alpha=0.3, axis='y')

# Execution time
times = [r.total_time_ms for r in all_results]
axes[1].bar(task_names, times, color='steelblue', alpha=0.7, edgecolor='black')
axes[1].set_title('Total Execution Time', fontsize=12, fontweight='bold')
axes[1].set_ylabel('Time (ms)')
axes[1].grid(True, alpha=0.3, axis='y')

# Step-by-step trace for fibonacci
fib_result = all_results[0]
step_nums = range(1, len(fib_result.steps) + 1)
step_pass = [1 if s.result and s.result.passed else 0 for s in fib_result.steps]

axes[2].bar(step_nums, step_pass, color=['red' if p == 0 else 'green' for p in step_pass],
            alpha=0.7, edgecolor='black')
axes[2].set_title('Fibonacci: Step-by-Step Trace', fontsize=12, fontweight='bold')
axes[2].set_xlabel('Attempt')
axes[2].set_ylabel('Pass (1) / Fail (0)')
axes[2].set_yticks([0, 1])
axes[2].set_yticklabels(['Fail', 'Pass'])

plt.tight_layout()
plt.show()

print("\nKey insight: Self-debugging turns a 0% one-shot pass into 100% multi-attempt pass.")

---

## 6. Part 5: Code Review Agent

A specialized agent that reviews code for bugs, style issues, and improvement opportunities.

### FAANG Interview Question

**Q: "How would you build an automated code review agent?"**

**A**: Three-layer review: (1) **Static analysis** — AST-based checks for unused variables, unreachable code, complexity. (2) **Style checks** — naming conventions, line length, missing docstrings. (3) **Semantic review** — LLM-based understanding of logic correctness, edge cases, security issues. The agent produces structured findings (severity: critical/high/medium/low) with specific line numbers and fix suggestions.

In [ ]:
@dataclass
class ReviewFinding:
    """A single code review finding."""
    severity: str  # critical, high, medium, low
    category: str  # bug, style, performance, security
    message: str
    line: int = 0
    suggestion: str = ""


class CodeReviewAgent:
    """Automated code review using AST analysis."""
    
    def review(self, source_code: str) -> List[ReviewFinding]:
        """Run all review checks on source code."""
        findings = []
        findings.extend(self._check_style(source_code))
        findings.extend(self._find_bugs(source_code))
        findings.extend(self._check_complexity(source_code))
        # Sort by severity
        severity_order = {'critical': 0, 'high': 1, 'medium': 2, 'low': 3}
        findings.sort(key=lambda f: severity_order.get(f.severity, 4))
        return findings
    
    def _check_style(self, code: str) -> List[ReviewFinding]:
        """Check naming conventions and style."""
        findings = []
        try:
            tree = ast.parse(code)
        except SyntaxError as e:
            return [ReviewFinding('critical', 'bug', f'Syntax error: {e}', 
                                  line=e.lineno or 0)]
        
        for node in ast.walk(tree):
            # Check function naming (should be snake_case)
            if isinstance(node, ast.FunctionDef):
                if not re.match(r'^[a-z_][a-z0-9_]*$', node.name) and node.name != '__init__':
                    findings.append(ReviewFinding(
                        'low', 'style',
                        f'Function "{node.name}" should be snake_case',
                        line=node.lineno,
                        suggestion=re.sub(r'([A-Z])', r'_\1', node.name).lower().strip('_')
                    ))
                # Check for missing docstring
                if not ast.get_docstring(node):
                    findings.append(ReviewFinding(
                        'low', 'style',
                        f'Function "{node.name}" missing docstring',
                        line=node.lineno
                    ))
        
        # Check line length
        for i, line in enumerate(code.splitlines(), 1):
            if len(line) > 100:
                findings.append(ReviewFinding(
                    'low', 'style', f'Line too long ({len(line)} > 100 chars)',
                    line=i
                ))
        
        return findings
    
    def _find_bugs(self, code: str) -> List[ReviewFinding]:
        """Find potential bugs using AST analysis."""
        findings = []
        try:
            tree = ast.parse(code)
        except SyntaxError:
            return findings
        
        for node in ast.walk(tree):
            # Bare except (catches everything including KeyboardInterrupt)
            if isinstance(node, ast.ExceptHandler) and node.type is None:
                findings.append(ReviewFinding(
                    'high', 'bug',
                    'Bare except clause catches all exceptions',
                    line=node.lineno,
                    suggestion='Use "except Exception:" instead'
                ))
            
            # Mutable default arguments
            if isinstance(node, ast.FunctionDef):
                for default in node.args.defaults:
                    if isinstance(default, (ast.List, ast.Dict, ast.Set)):
                        findings.append(ReviewFinding(
                            'high', 'bug',
                            f'Mutable default argument in "{node.name}"',
                            line=node.lineno,
                            suggestion='Use None and initialize inside function'
                        ))
        
        return findings
    
    def _check_complexity(self, code: str) -> List[ReviewFinding]:
        """Flag overly complex functions."""
        findings = []
        try:
            tree = ast.parse(code)
        except SyntaxError:
            return findings
        
        for node in ast.walk(tree):
            if isinstance(node, ast.FunctionDef):
                # Count decision points in this function
                complexity = 1
                for child in ast.walk(node):
                    if isinstance(child, (ast.If, ast.For, ast.While, ast.ExceptHandler)):
                        complexity += 1
                
                if complexity > 10:
                    findings.append(ReviewFinding(
                        'medium', 'performance',
                        f'Function "{node.name}" has high complexity ({complexity})',
                        line=node.lineno,
                        suggestion='Consider breaking into smaller functions'
                    ))
        
        return findings


# Demo code review
review_code = '''
def processData(items, cache={}):
    result = []
    for item in items:
        try:
            if item > 0:
                if item < 100:
                    if item not in cache:
                        cache[item] = item * 2
                    result.append(cache[item])
        except:
            pass
    return result
'''

reviewer = CodeReviewAgent()
findings = reviewer.review(review_code)

print("Code Review Results:")
print("=" * 60)
for f in findings:
    print(f"  [{f.severity.upper()}] [{f.category}] Line {f.line}: {f.message}")
    if f.suggestion:
        print(f"    Fix: {f.suggestion}")

print(f"\nTotal findings: {len(findings)}")
print(f"  Critical: {sum(1 for f in findings if f.severity == 'critical')}")
print(f"  High:     {sum(1 for f in findings if f.severity == 'high')}")
print(f"  Medium:   {sum(1 for f in findings if f.severity == 'medium')}")
print(f"  Low:      {sum(1 for f in findings if f.severity == 'low')}")

---

## 7. Part 6: Plan-and-Solve Coding Agent

For complex tasks, ReAct (think-act-observe) isn't enough. Plan-and-Solve decomposes the task first.

### FAANG Interview Question

**Q: "When would you use Plan-and-Solve vs ReAct for a coding task?"**

**A**: **ReAct** for single-function tasks where the path is clear (implement fibonacci, fix a bug). **Plan-and-Solve** for multi-step tasks that require coordination (build a REST API with 3 endpoints, refactor a module). Plan-and-Solve prevents the agent from getting lost in details — it commits to a plan first, then executes step by step. The risk is that the plan might be wrong; good agents can revise the plan mid-execution.

In [ ]:
class PlanAndSolveAgent:
    """Coding agent that decomposes tasks into steps before executing.
    
    Steps:
    1. Analyze the task and create a plan
    2. Execute each step of the plan
    3. Integrate results
    4. Verify with tests
    """
    
    def __init__(self, sandbox: CodeSandbox):
        self.sandbox = sandbox
    
    def _create_plan(self, task: str) -> List[Dict[str, str]]:
        """Create an execution plan for the task.
        
        In production: LLM generates this plan.
        Here: template-based for demo.
        """
        # Simple plan templates based on task keywords
        if 'class' in task.lower() or 'data structure' in task.lower():
            return [
                {"step": "Define the class with __init__", "type": "structure"},
                {"step": "Implement core methods", "type": "logic"},
                {"step": "Add helper methods", "type": "logic"},
                {"step": "Add error handling", "type": "safety"},
            ]
        return [
            {"step": "Write function signature with docstring", "type": "structure"},
            {"step": "Implement core logic", "type": "logic"},
            {"step": "Handle edge cases", "type": "safety"},
        ]
    
    def _execute_plan(self, task: str, plan: List[Dict]) -> str:
        """Execute the plan step by step.
        
        In production: each step calls the LLM with context.
        Here: returns pre-built solution.
        """
        # Demo: return a complete Stack implementation
        if 'stack' in task.lower():
            return textwrap.dedent('''
            class Stack:
                """A stack data structure with push, pop, peek, and size."""
                
                def __init__(self):
                    self._items = []
                
                def push(self, item):
                    """Push item onto the stack."""
                    self._items.append(item)
                
                def pop(self):
                    """Remove and return top item. Raises IndexError if empty."""
                    if self.is_empty():
                        raise IndexError("Pop from empty stack")
                    return self._items.pop()
                
                def peek(self):
                    """Return top item without removing. Raises IndexError if empty."""
                    if self.is_empty():
                        raise IndexError("Peek at empty stack")
                    return self._items[-1]
                
                def is_empty(self):
                    """Check if stack is empty."""
                    return len(self._items) == 0
                
                def size(self):
                    """Return number of items."""
                    return len(self._items)
            ''').strip() + '\n'
        
        return "# Plan execution not implemented for this task\npass\n"
    
    def solve(self, task: str, test_code: str) -> AgentResult:
        """Solve a task using plan-and-solve approach."""
        start = time.time()
        steps = []
        
        # Step 1: Create plan
        plan = self._create_plan(task)
        plan_str = "\n".join(f"  {i+1}. {s['step']}" for i, s in enumerate(plan))
        steps.append(AgentStep(
            thought=f"Created plan with {len(plan)} steps:\n{plan_str}",
            action="plan", code=""
        ))
        
        # Step 2: Execute plan
        code = self._execute_plan(task, plan)
        
        # Step 3: Test
        exec_result = self.sandbox.execute_with_tests(code, test_code)
        steps.append(AgentStep(
            thought="Testing implementation",
            action="test", code=code, result=exec_result
        ))
        
        total_time = (time.time() - start) * 1000
        return AgentResult(
            code=code, passed=exec_result.passed,
            steps=steps, attempts=1, total_time_ms=total_time
        )


# Demo Plan-and-Solve
pas_agent = PlanAndSolveAgent(sandbox=CodeSandbox(timeout=5))

stack_tests = textwrap.dedent('''
s = Stack()
assert s.is_empty() == True
s.push(1)
s.push(2)
s.push(3)
assert s.size() == 3
assert s.peek() == 3
assert s.pop() == 3
assert s.size() == 2
assert s.pop() == 2
assert s.pop() == 1
assert s.is_empty() == True
try:
    s.pop()
    assert False, "Should have raised IndexError"
except IndexError:
    pass
print("All Stack tests passed!")
''')

result = pas_agent.solve("Implement a Stack data structure", stack_tests)
print("Plan-and-Solve Agent Demo")
print("=" * 60)
for step in result.steps:
    print(f"\n[{step.action.upper()}] {step.thought}")
    if step.result:
        print(f"  Result: {'PASS' if step.result.passed else 'FAIL'}")
        if step.result.stdout:
            print(f"  Output: {step.result.stdout.strip()}")

print(f"\nFinal: {'PASS' if result.passed else 'FAIL'} in {result.total_time_ms:.0f}ms")

---

## 8. Part 7: Multi-Turn Context Management

Real coding agents work over multiple turns, accumulating files and history.

In [ ]:
class CodingConversation:
    """Manages multi-turn context for coding agents.
    
    Tracks files, conversation history, and provides
    context truncation for long conversations.
    """
    
    def __init__(self, max_context_chars: int = 8000):
        self.files: Dict[str, str] = {}
        self.history: List[Dict[str, str]] = []
        self.max_context = max_context_chars
    
    def add_file(self, filename: str, content: str):
        """Add or update a file in the workspace."""
        self.files[filename] = content
    
    def add_turn(self, role: str, message: str, code_changes: Optional[Dict] = None):
        """Add a conversation turn."""
        turn = {"role": role, "message": message}
        if code_changes:
            turn["code_changes"] = code_changes
            for filename, content in code_changes.items():
                self.files[filename] = content
        self.history.append(turn)
    
    def get_context(self) -> str:
        """Build context string with smart truncation.
        
        Strategy: Always include current files + recent history.
        Truncate oldest history first.
        """
        # Files section (always included)
        files_section = "=== Current Files ===\n"
        for name, content in self.files.items():
            files_section += f"--- {name} ---\n{content}\n"
        
        # History section (truncate from oldest)
        remaining = self.max_context - len(files_section)
        history_section = "\n=== Conversation ===\n"
        
        # Include history from most recent
        included = []
        for turn in reversed(self.history):
            turn_str = f"[{turn['role']}]: {turn['message']}\n"
            if len(turn_str) + sum(len(t) for t in included) < remaining:
                included.insert(0, turn_str)
            else:
                break
        
        if len(included) < len(self.history):
            history_section += f"... ({len(self.history) - len(included)} earlier turns omitted) ...\n"
        history_section += "".join(included)
        
        return files_section + history_section


# Demo multi-turn conversation
conv = CodingConversation(max_context_chars=2000)

# Turn 1: User asks for a function
conv.add_turn("user", "Write a function to check if a number is prime")
conv.add_turn("agent", "Here's the implementation:", 
              {"math_utils.py": "def is_prime(n):\n    if n < 2:\n        return False\n    for i in range(2, int(n**0.5) + 1):\n        if n % i == 0:\n            return False\n    return True\n"})

# Turn 2: User asks for tests
conv.add_turn("user", "Add tests for this function")
conv.add_turn("agent", "Added test file:",
              {"test_math.py": "from math_utils import is_prime\n\nassert is_prime(2) == True\nassert is_prime(17) == True\nassert is_prime(4) == False\nassert is_prime(1) == False\nprint('All tests passed!')\n"})

# Turn 3: User asks for optimization
conv.add_turn("user", "Can you optimize it for large numbers?")

print("Multi-Turn Context:")
print("=" * 60)
print(conv.get_context())
print(f"\nFiles tracked: {list(conv.files.keys())}")
print(f"Turns in history: {len(conv.history)}")

---

## 9. Part 8: Coding Agent Performance Metrics

How do we measure if a coding agent is good? (Cross-reference Tutorial 43 for detailed metrics.)

In [ ]:
from math import comb

def pass_at_k(n, c, k):
    """Unbiased pass@k estimator (from Tutorial 43)."""
    if n - c < k:
        return 1.0
    return 1.0 - comb(n - c, k) / comb(n, k)


class CodingAgentBenchmark:
    """Benchmark a coding agent on a set of tasks."""
    
    def __init__(self, agent, n_samples: int = 3):
        self.agent = agent
        self.n_samples = n_samples
    
    def run_benchmark(self, tasks: List[Tuple[str, str]]) -> Dict:
        """Run agent on all tasks multiple times.
        
        Args:
            tasks: list of (task_description, test_code) pairs
        Returns:
            Dictionary with pass@k metrics and per-task results
        """
        all_results = []
        
        for task_desc, test_code in tasks:
            task_passes = 0
            task_attempts = []
            
            for _ in range(self.n_samples):
                result = self.agent.solve(task_desc, test_code)
                if result.passed:
                    task_passes += 1
                task_attempts.append(result.attempts)
            
            all_results.append({
                "task": task_desc,
                "n_correct": task_passes,
                "n_total": self.n_samples,
                "avg_attempts": np.mean(task_attempts),
                "pass_at_1": pass_at_k(self.n_samples, task_passes, 1),
            })
        
        # Aggregate metrics
        overall_pass_1 = np.mean([r["pass_at_1"] for r in all_results])
        avg_attempts = np.mean([r["avg_attempts"] for r in all_results])
        solve_rate = np.mean([1 if r["n_correct"] > 0 else 0 for r in all_results])
        
        return {
            "pass_at_1": overall_pass_1,
            "solve_rate": solve_rate,
            "avg_attempts": avg_attempts,
            "per_task": all_results
        }


# Run benchmark
benchmark = CodingAgentBenchmark(agent=agent, n_samples=3)
bench_results = benchmark.run_benchmark(tasks)

print("Coding Agent Benchmark Results")
print("=" * 50)
print(f"Overall pass@1:    {bench_results['pass_at_1']:.2%}")
print(f"Solve rate:        {bench_results['solve_rate']:.2%}")
print(f"Avg attempts:      {bench_results['avg_attempts']:.1f}")

print("\nPer-task results:")
for r in bench_results['per_task']:
    print(f"  {r['task']}: {r['n_correct']}/{r['n_total']} correct, pass@1={r['pass_at_1']:.2%}")

# Visualization
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

task_labels = [r['task'] for r in bench_results['per_task']]
pass_rates = [r['pass_at_1'] for r in bench_results['per_task']]
avg_attempts_per_task = [r['avg_attempts'] for r in bench_results['per_task']]

axes[0].bar(task_labels, pass_rates, color='steelblue', alpha=0.7, edgecolor='black')
axes[0].set_title('pass@1 by Task', fontsize=12, fontweight='bold')
axes[0].set_ylabel('pass@1')
axes[0].set_ylim(0, 1.1)
axes[0].grid(True, alpha=0.3, axis='y')

axes[1].bar(task_labels, avg_attempts_per_task, color='coral', alpha=0.7, edgecolor='black')
axes[1].set_title('Average Attempts by Task', fontsize=12, fontweight='bold')
axes[1].set_ylabel('Avg Attempts')
axes[1].grid(True, alpha=0.3, axis='y')

plt.tight_layout()
plt.show()

---

## 10. FAANG Interview Questions

### Q1: "Design a production coding agent architecture."

**A**: Four layers:
1. **Orchestrator**: Manages conversation state, routes to tools, handles retries.
2. **Tool layer**: File I/O, terminal, search, test runner — each with safety validation.
3. **Execution layer**: Sandboxed code execution with resource limits, output capture.
4. **Memory layer**: File tracking, conversation history with smart truncation, previous solutions.

Key design decisions: max iterations (prevent loops), context window management (what to include vs truncate), error parsing (extract actionable info from stack traces), cost tracking (token usage per attempt).

---

### Q2: "How do you sandbox untrusted code execution?"

**A**: Defense in depth:
1. **Pre-execution**: Static analysis — block imports of dangerous modules, check for shell access patterns.
2. **Process isolation**: Subprocess with timeout. Never run in the agent's process.
3. **Resource limits**: CPU time (5-30s), memory (256MB-1GB), disk writes (10MB).
4. **Network**: Block all outbound connections (iptables/seccomp).
5. **Filesystem**: tmpfs mount, no access to host filesystem.
6. **Container**: For production, use Docker/gVisor with no capabilities.

---

### Q3: "Self-debugging vs one-shot generation: tradeoffs?"

**A**:
- **One-shot**: Faster (1 attempt), cheaper (1 LLM call), simpler. Works for easy tasks.
- **Self-debugging**: Higher pass rate (research shows 2-3x improvement), handles edge cases, learns from errors. But: more expensive (multiple LLM calls), risk of infinite loops, needs good error parsing.
- **Hybrid**: Generate one-shot first. If it passes, done. If not, enter debug loop. This gets 90%+ of the benefit at lower cost.

---

### Q4: "How would you train a coding agent with RL?"

**A**: Use GRPO (Tutorial 41) or PPO (Tutorial 40):
- **Reward**: Binary (+1 if tests pass, -0.1 per failed attempt). Can add code quality metrics.
- **State**: (task description, current code, error message, attempt number).
- **Action**: Generate next code version.
- **Training**: Sample K solutions per task (GRPO), score with test execution, update policy.
- **Key insight**: Test execution is a *perfect reward signal* — no reward model needed. This is why coding is ideal for RL.

---

### Q5: "Multi-agent code review system: how would you design it?"

**A**: Three specialized agents:
1. **Writer**: Generates code from spec.
2. **Reviewer**: Runs static analysis + LLM review. Produces structured findings.
3. **Tester**: Generates and runs test cases.

Workflow: Writer generates -> Reviewer critiques -> Writer revises -> Tester validates -> iterate. The Supervisor agent coordinates, tracks findings, and decides when quality is sufficient. Each agent has its own context and tools.

---

## 11. Key Takeaways

### Foundation (Know These Cold)
- [ ] AST parsing: `ast.parse()`, `ast.walk()`, node types (FunctionDef, ClassDef, Name, Call)
- [ ] Sandboxed execution: subprocess isolation, timeout, resource limits, blocked patterns
- [ ] Self-debugging loop: generate -> execute -> observe error -> fix -> repeat
- [ ] Tools: FileRead, FileWrite, Terminal, CodeSearch, TestRunner

### Implementation (Be Able to Code)
- [ ] `CodeAnalyzer` — extract functions, classes, complexity, call graph from source
- [ ] `CodeSandbox` — safe execution with subprocess + tempfile + timeout
- [ ] `SelfDebugAgent` — generate-test-fix loop with attempt tracking
- [ ] `CodeReviewAgent` — AST-based bug finding, style checking

### Architecture (Interview Differentiator)
- [ ] Plan-and-Solve for complex multi-step tasks
- [ ] Multi-turn context management with smart truncation
- [ ] Agent benchmarking with pass@k metrics
- [ ] Training coding agents with RL (GRPO + test execution as reward)

### Safety (Non-Negotiable)
- [ ] Never execute untrusted code in the agent's process
- [ ] Always use timeouts (agents can generate infinite loops)
- [ ] Block network access, dangerous imports, filesystem escapes
- [ ] Defense in depth: pre-check + process isolation + resource limits

---

**Next**: The `coding_agent/` project synthesizes all five tutorials (40-44) into a standalone, runnable coding agent.